In [42]:
import pandas as pd
import numpy as np
import re
import csv
import os
from evcouplings.compare import DistanceMap

## generate 3-to-1 letter AA codes

In [3]:
# Mapping from three-letter to one-letter amino acid codes
three_to_one = {
    'Ala': 'A', 'Cys': 'C', 'Asp': 'D', 'Glu': 'E', 'Phe': 'F', 'Gly': 'G', 'His': 'H',
    'Ile': 'I', 'Lys': 'K', 'Leu': 'L', 'Met': 'M', 'Asn': 'N', 'Pro': 'P', 'Gln': 'Q',
    'Arg': 'R', 'Ser': 'S', 'Thr': 'T', 'Val': 'V', 'Trp': 'W', 'Tyr': 'Y'
}

def convert_three_to_one(mutation):
    # Modify the regex to match the gene name but ignore it in the output
    match = re.match(r"[a-zA-Z0-9]+_p\.([A-Z][a-z]{2})(\d+)([A-Z][a-z]{2})", mutation)
    if not match:
        return None
    original_aa, position, new_aa = match.groups()
    try:
        # Return just the one-letter mutation string
        return f"p.{three_to_one[original_aa]}{position}{three_to_one[new_aa]}"
    except KeyError:
        return None

# Example usage:
print(convert_three_to_one("atpE_p.Asp28Gly"))  # Output: p.D28G
print(convert_three_to_one("atpE_p.Ala249Thr"))  # Output: p.A249T


p.D28G
p.A249T


In [4]:
# check if the genes of interest are in catalog df
protein_sequences_file = '/project/pi_annagreen_umass_edu/mahbuba/all_projects/resistance_forecast/data/catalog/protein_sequences.csv'
protein_sequences_df = pd.read_csv(protein_sequences_file)
# Extract genes of interest
genes_of_interest = protein_sequences_df['gene'].unique()

In [5]:
## load the 2023 complete excel file

# Load the WHO catalog
who_catalog_path = '/project/pi_annagreen_umass_edu/mahbuba/all_projects/resistance_forecast/data/catalog/WHO-UCN-TB-2023.7-eng.xlsx'  # Update the path as necessary
who_catalog = pd.read_excel(who_catalog_path, sheet_name='Catalogue_master_file',header=2)

/work/pi_annagreen_umass_edu/mahbuba/esmfold/lib/python3.10/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)


In [20]:

# Step 1: Select the columns with the correct names
catalog_df = who_catalog[[
    'drug', 
    'gene',
    'variant', 
    'Present_R', 
    'Present_S', 
    'FINAL CONFIDENCE GRADING'
]]


# Step 2: Rename the columns to more meaningful names (optional)
catalog_df.columns = ['drug','gene','variant', 'Present_R', 'Present_S', 'FINAL_CONFIDENCE_GRADING']

# Step 3: Check the first few rows of the selected data
print(catalog_df.head())
# Extract relevant columns

# Filter out rows with NaN values in the 'drug' or 'variant' columns
catalog_df = catalog_df.dropna(subset=['drug', 'variant'])

# subset for target genes
filtered_df = catalog_df[catalog_df['gene'].str.startswith(tuple(genes_of_interest))]


       drug  gene         variant  Present_R  Present_S  \
0  Amikacin  bacA   bacA_c.102G>A        0.0        1.0   
1  Amikacin  bacA  bacA_c.1044G>A       91.0     1112.0   
2  Amikacin  bacA   bacA_c.105C>G        0.0        1.0   
3  Amikacin  bacA  bacA_c.1065T>G        0.0        3.0   
4  Amikacin  bacA  bacA_c.1080G>A        0.0        2.0   

     FINAL_CONFIDENCE_GRADING  
0  4) Not assoc w R - Interim  
1            5) Not assoc w R  
2  4) Not assoc w R - Interim  
3  4) Not assoc w R - Interim  
4  4) Not assoc w R - Interim  


In [21]:
# Display the filtered DataFrame
print("number of isolates before discarding invalid residues", len(filtered_df))
# Discard rows that have 'ins' or 'del' in the variant column
filtered_df = filtered_df[~filtered_df['variant'].str.contains('ins|del')]
filtered_df=filtered_df.rename(columns={"FINAL_CONFIDENCE_GRADING": "confidence"})
print("number of isolates after discarding invalid residues", len(filtered_df))

number of isolates before discarding invalid residues 11280
number of isolates after discarding invalid residues 11051


In [ ]:
## apply the 3-to-1 conversion
filtered_df['one_letter_mutation'] = filtered_df['variant'].apply(convert_three_to_one)
# drop invalid mutations
# checked all variants to ensure they match the format A321B.
filtered_df = filtered_df.dropna(subset=['one_letter_mutation'])
filtered_df =filtered_df.drop_duplicates()

In [29]:
filtered_df.to_csv('../data/derived_features/2023_mutations_with_one_letter_all_confidence.csv', index=False)

## feature 1: frequency

In [ ]:
filtered_df['frequency'] = (filtered_df['Present_R']) / (filtered_df['Present_R'] + filtered_df['Present_S'])
filtered_df.drop(columns=['Present_R','Present_S'], inplace=True)

In [33]:
filtered_df.to_csv('../data/derived_features/2023_all_proteins_frequency_catalog.csv', index=False)

## feature 2: proximity to nearest r-conferring mutations

In [38]:
def extract_position_from_mutation(mutation):
    ## to extract position from mutation column
    match = re.search(r'\d+', mutation)
    if match:
        return int(match.group(0))
    return None

def adjust_number(index):
    # adjust index offset between pdb and who indices for rpob
    return index + 6 


def proximity_1D(gene_subset, unique_valid_r_positions, gene_name):
    proximity_to_nonself_r_conferring = []
    positions_causing_error = []
    nearest_mutation_index = []
    for index, row in gene_subset.iterrows():
        #take the WHO adjusted position wrt PDB for rpoB and position for all 
        if gene_name=='rpoB':
            current_position = row['WHO_Adjusted_Position']
        else:
            current_position= row['position']
        distances = []

        for r_pos in unique_valid_r_positions:
            # iterate through list of R positions we have to compute against the current position
            if r_pos != current_position: #if its a non-self R residue (which we expect)

                try:
                    dist = abs(current_position - r_pos)
                    if not np.isnan(dist):
                        # if distance is not NaN, add it to the dist dict with the current R residue position
                        distances.append((dist, r_pos))
                except KeyError:
                    #keep a log of positions causing Nan dist value despite taking the diff between self and nonself R positions
                    positions_causing_error.append((current_position, r_pos))
                    continue

        if distances:
            #take the min of all distances for per residue
            min_distance, min_index = min(distances, key=lambda x: x[0])
        else:
            min_distance, min_index = np.nan, np.nan
        # Append results to the lists so that we can just add this list as a column to the dataframe
        proximity_to_nonself_r_conferring.append(min_distance)
        nearest_mutation_index.append(min_index)
    # Store the results in the phenotype_data DataFrame
    gene_subset['Proximity_1D'] = proximity_to_nonself_r_conferring
    gene_subset['Nearest_1D_Index'] = nearest_mutation_index
    return gene_subset    

def non_self_proximity_r_mutants(gene_subset, unique_valid_r_positions, distance_map, gene_name):
    positions_causing_error = []
    proximity_to_nonself_r_conferring = []
    nearest_mutation_index = []

    for index, row in gene_subset.iterrows():
        #take the WHO adjusted position wrt PDB for rpoB and position for all 
        if gene_name=='rpoB':
            current_position = row['WHO_Adjusted_Position']
        else:
            current_position= row['position']
        distances = []

        for r_pos in unique_valid_r_positions:
            # iterate through list of R positions we have to compute against the current position
            if r_pos != current_position: #if its a non-self R residue (which we expect)

                try:
                    dist = distance_map.dist(current_position, r_pos, raise_na=True)
                    if not np.isnan(dist):
                        # if distance is not NaN, add it to the dist dict with the current R residue position
                        distances.append((dist, r_pos))
                except KeyError:
                    #keep a log of positions causing Nan dist value despite taking the diff between self and nonself R positions
                    positions_causing_error.append((current_position, r_pos))
                    continue

        if distances:
            #take the min of all distances for per residue
            min_distance, min_index = min(distances, key=lambda x: x[0])
        else:
            min_distance, min_index = np.nan, np.nan
        # Append results to the lists so that we can just add this list as a column to the dataframe
        proximity_to_nonself_r_conferring.append(min_distance)
        nearest_mutation_index.append(min_index)
    # Store the results in the phenotype_data DataFrame
    gene_subset['Proximity_to_R_Conferring'] = proximity_to_nonself_r_conferring
    gene_subset['Nearest_Mutation_Index'] = nearest_mutation_index
    return gene_subset
# Map confidence levels to phenotypes
def map_confidence_to_phenotype(confidence):
    if confidence in ['1) Assoc w R', '2) Assoc w R - Interim']:
        return 'Resistant'
    elif confidence in ['4) Not assoc w R - Interim', '5) Not assoc w R']:
        return 'Susceptible'
    else:
        return 'Unknown'

## Compute proximity

In [39]:
#load the derived dataframe from previous step
catalog_df=pd.read_csv('../data/derived_features/2023_all_proteins_frequency_catalog.csv')
catalog_df['phenotype'] =catalog_df['confidence'].apply(map_confidence_to_phenotype)
print("distribution of phenotype", catalog_df['phenotype'].value_counts())
# extract position from mutation and create a column
catalog_df['position'] =catalog_df['one_letter_mutation'].str.extract(r'(\d+)').astype(int)
#Get the unique combinations of gene and drug
unique_genes_drugs = catalog_df[['gene', 'drug']].drop_duplicates()

# Print each gene and drug pair - debug step
for index, row in unique_genes_drugs.iterrows():
    print(f"Gene: {row['gene']}, Drug: {row['drug']}")
#create list of unique gene names
gene_names=list(np.unique(catalog_df['gene']))

distribution of phenotype phenotype
Unknown        5699
Resistant       411
Susceptible      62
Name: count, dtype: int64
Gene: atpE, Drug: Bedaquiline
Gene: pepQ, Drug: Bedaquiline
Gene: Rv0678, Drug: Bedaquiline
Gene: tlyA, Drug: Capreomycin
Gene: fgd1, Drug: Clofazimine
Gene: pepQ, Drug: Clofazimine
Gene: Rv0678, Drug: Clofazimine
Gene: ddn, Drug: Delamanid
Gene: fgd1, Drug: Delamanid
Gene: embB, Drug: Ethambutol
Gene: ethA, Drug: Ethionamide
Gene: inhA, Drug: Ethionamide
Gene: inhA, Drug: Isoniazid
Gene: katG, Drug: Isoniazid
Gene: gyrA, Drug: Levofloxacin
Gene: gyrB, Drug: Levofloxacin
Gene: rplC, Drug: Linezolid
Gene: gyrA, Drug: Moxifloxacin
Gene: gyrB, Drug: Moxifloxacin
Gene: pncA, Drug: Pyrazinamide
Gene: rpoB, Drug: Rifampicin
Gene: gid, Drug: Streptomycin
Gene: rpsL, Drug: Streptomycin


In [40]:
##load the protein details file - we get the dist map id from it
protein_details_path = '../data/catalog/17_proteins_details.xlsx'  # Update the path as necessary
protein_details = pd.read_excel(protein_details_path, sheet_name='Sheet1')
# Filter the DataFrame for the genes of interest
filtered_df = protein_details[protein_details['gene_name'].str.contains('|'.join(gene_names), case=False, na=False)]
print("previewing protein details file")
filtered_df.head()

previewing protein details file


,Drug,gene_name,filename,Uniprot,references,Entry,Protein names,rv,drug_full,Aliases,start_position_on_the_genomic_accession,end_position_on_the_genomic_accession,orientation
0,RIF,rpoB,rpoBC.fasta,RPOB_MYCTU,"WHO 2021 catalog, category 1",P9WGY9,DNA-directed RNA polymerase subunit beta (RNAP...,Rv0667,rifampicin,Rv0667,759806.0,763324.0,plus
1,INH,inhA,FabG1-inhA.fasta,INHA_MYCTU,"WHO 2021 catalog, category 1",P9WGR1,Enoyl-[acyl-carrier-protein] reductase [NADH] ...,Rv1484,isoniazid,Rv1484,1674201.0,1675010.0,plus
2,INH,katG,KatG.fasta,KATG_MYCTU,"WHO 2021 catalog, category 1",P9WIE5,Catalase-peroxidase (CP) (EC 1.11.1.21) (Perox...,Rv1908c,isoniazid,Rv1908c,2153889.0,2156111.0,minus
3,EMB,embB,embCAB.fasta,EMBB_MYCTU,"WHO 2021 catalog, category 1",P9WNL7,Probable arabinosyltransferase B (EC 2.4.2.-),Rv3795,ethambutol,Rv3795,4246513.0,4249809.0,plus
4,PZA,pncA,pncA.fasta,PNCA_MYCTU,"WHO 2021 catalog, category 1",I6XD65,Nicotinamidase/pyrazinamidase (Nicotinamidase)...,Rv2043c,pyrazinamide,Rv2043c,2288681.0,2289241.0,minus


In [43]:
# Initialize an empty list to store all processed DataFrames
all_gene_data = []


for index, row in filtered_df.iterrows():
    # Extract metadata from current row
    fasta_filename = row['filename']
    uniprot = row['Uniprot']
    entry = row['Entry']
    drug = row['drug_full']
    drug_code = row['Drug']
    gene_name = row['gene_name']
    
    print(f"\nProcessing {gene_name} from file: {fasta_filename}")

    # Subset catalog to this gene
    gene_subset = catalog_df[catalog_df['gene'] == gene_name].drop_duplicates()

    # Load distance map
    distmap_path = f"/project/pi_annagreen_umass_edu/mahbuba/all_projects/resistance_forecast/data/distmaps/{uniprot}/{entry}"
    dist_map = DistanceMap.from_file(distmap_path)
    gene_length = dist_map.dist_matrix.shape[0]
    print(f"Distance map length for {gene_name}: {gene_length}")

    # Pull residues
    residues_df = dist_map.residues_i.copy()
    residues_df['coord_id'] = residues_df['coord_id'].astype(int)
    
    # Get WHO mutation info
    gene_subset = gene_subset.copy()
    #apply adjust number method if it is rpob
    if gene_name == 'rpoB':
        gene_subset['WHO_Adjusted_Position'] = gene_subset['position'].apply(adjust_number)
        #set the who adjusted position as r-conferring positions array for proximity calculation
        r_conferring_positions = gene_subset[gene_subset['phenotype'] == 'Resistant']['WHO_Adjusted_Position'].tolist()
    else:
        r_conferring_positions = gene_subset[gene_subset['phenotype'] == 'Resistant']['position'].tolist()
    

    unique_valid_r_positions = np.unique(r_conferring_positions)
    print("Number of Unique valid R-conferring positions:", len(unique_valid_r_positions))

    #proximity -1d
    gene_subset =  proximity_1D(gene_subset, unique_valid_r_positions, gene_name)
    

    # Proximity analysis-proximity 3d
    gene_subset = non_self_proximity_r_mutants(gene_subset, unique_valid_r_positions, dist_map, gene_name)


    # load and merge G-scores
    try:
        g_scores = pd.read_csv(f"/project/pi_annagreen_umass_edu/mahbuba/all_projects/resistance_forecast/data/distmaps/{uniprot}/G_scores.csv")
        print(f"G_scores loaded for {uniprot}")
        if gene_name == 'rpoB':
            # gene_subset['adjusted_position'] = gene_subset['position'].apply(adjust_number)
            merged = pd.merge(gene_subset, g_scores, left_on='WHO_Adjusted_Position', right_on='residue', how='left')
        else:
            #since for other genes we are not adjusting their position, we use the actual column
            merged = pd.merge(gene_subset, g_scores, left_on='position', right_on='residue', how='left')

        merged.drop(columns=['residue', 'Unnamed: 0'], errors='ignore', inplace=True)

        all_gene_data.append(merged)

    except FileNotFoundError as e:
        print(f"Skipping G-score merge for {uniprot}: {e}")
        all_gene_data.append(gene_subset)



Processing rpoB from file: rpoBC.fasta
Distance map length for rpoB: 1138
Number of Unique valid R-conferring positions: 26
G_scores loaded for RPOB_MYCTU

Processing inhA from file: FabG1-inhA.fasta
Distance map length for inhA: 246
Number of Unique valid R-conferring positions: 1
G_scores loaded for INHA_MYCTU

Processing katG from file: KatG.fasta
Distance map length for katG: 716
Number of Unique valid R-conferring positions: 2
G_scores loaded for KATG_MYCTU

Processing embB from file: embCAB.fasta
Distance map length for embB: 1054
Number of Unique valid R-conferring positions: 6
G_scores loaded for EMBB_MYCTU

Processing pncA from file: pncA.fasta
Distance map length for pncA: 185
Number of Unique valid R-conferring positions: 95
G_scores loaded for PNCA_MYCTU

Processing gyrA from file: gyrBA.fasta
Distance map length for gyrA: 766
Number of Unique valid R-conferring positions: 5
G_scores loaded for GYRA_MYCTU

Processing rpsL from file: rpsL.fasta
Distance map length for rpsL:

In [44]:
# After the loop, concatenate all the DataFrames in the list into a single large DataFrame
final_df = pd.concat(all_gene_data, ignore_index=True)

In [45]:
final_df.to_csv("../data/derived_features/2023_all_proteins_freq_proximity_gscore.csv",index=False)

In [46]:
final_df.head()

,drug,gene,variant,confidence,one_letter_mutation,phenotype,position,WHO_Adjusted_Position,Proximity_1D,Nearest_1D_Index,Proximity_to_R_Conferring,Nearest_Mutation_Index,G_score
0,Rifampicin,rpoB,rpoB_p.Ala1002Pro,3) Uncertain significance,p.A1002P,Unknown,1002,1008.0,511.0,497.0,31.751875,444.0,-0.257180
1,Rifampicin,rpoB,rpoB_p.Ala1055Pro,3) Uncertain significance,p.A1055P,Unknown,1055,1061.0,564.0,497.0,16.447369,443.0,0.289620
2,Rifampicin,rpoB,rpoB_p.Ala1072Gly,3) Uncertain significance,p.A1072G,Unknown,1072,1078.0,581.0,497.0,36.136792,458.0,-0.135454
3,Rifampicin,rpoB,rpoB_p.Ala10Thr,3) Uncertain significance,p.A10T,Unknown,10,16.0,160.0,176.0,NaN,NaN,NaN
4,Rifampicin,rpoB,rpoB_p.Ala1153Val,3) Uncertain significance,p.A1153V,Unknown,1153,1159.0,662.0,497.0,50.801629,432.0,-0.324753
